# Architecture Refinements: from "it runs" to "it trains well"

> **Previous recap**: In Part 4 we built a MiniGPT using the classic Transformer recipe:
> - Residual after attention, then LayerNorm (**Post-Norm**)
> - ReLU in the FFN (**ReLU FFN**)
> - LayerNorm for normalization (**LayerNorm**)
>
> **But modern LLMs (LLaMA, GPT-4, DeepSeek) use a different recipe.**
>
> Goal for this part: **upgrade the Part-4 Transformer block step by step into the modern LLM-style block, and understand *why* each change helps.**

We will make three upgrades. For each upgrade we ask three questions:
1. What is the old thing? How is it computed?
2. What is the problem with the old thing?
3. What is the new thing? Why is it better?

**We will use tiny numeric examples and compute by hand, so every step is trackable.**


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

### 0. Quick recap: what does the Part-4 block look like?

Let's paste the block you already know from Part 4, and modify it one change at a time.


In [ ]:
# === 这就是 Part 4 的版本，我们接下来的「改造对象」 ===

class FeedForward_Old(nn.Module):
    """原始 FFN：两layer Linear，中间 ReLU"""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


class TransformerBlock_Old(nn.Module):
    """Post-Norm + LayerNorm + ReLU FFN（Part 4 版本）"""
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ffn = FeedForward_Old(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)  # ← 普通 LayerNorm
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        # Post-Norm: 先算子layer，再 +residual，最后 Norm
        x = self.norm1(x + self.attention(x, x, x, need_weights=False)[0])
        x = self.norm2(x + self.ffn(x))
        return x

print("[ok] 这是 Part 4 的版本。接下来我们逐个升级。")
print("升级路线: LayerNorm->RMSNorm -> ReLU->SwiGLU -> Post-Norm->Pre-Norm")

---

### 1. Upgrade 1: LayerNorm -> RMSNorm

#### 1.1 What does LayerNorm compute? Do it by hand with 4 numbers

Assume a token vector is `[1, 3, 5, 7]` (4 dims, matching our Part-4 `d_model=4`).

**LayerNorm** makes the mean 0 and the standard deviation 1.

A useful analogy: grading on a curve. No matter how high or low raw scores are, after normalization the class average becomes 0 and the spread becomes consistent.

Steps:

```
1. mean      mu = (x1 + x2 + x3 + x4) / 4
2. variance  sigma^2 = ((x1-mu)^2 + ... + (x4-mu)^2) / 4
3. std       sigma = sqrt(sigma^2)
4. normalize x' = (x - mu) / sigma
5. affine    y = gamma * x' + beta
```

`gamma` and `beta` are learnable parameters.

Next we compute LayerNorm for `[1, 3, 5, 7]` by hand.


In [ ]:
# === LayerNorm 手工compute ===
print("=== LayerNorm Compute by hand：input x = [1, 3, 5, 7] ===")
print()

x = torch.tensor([1.0, 3.0, 5.0, 7.0])

# Step 1: mean
mu = x.mean()
print(f"Step 1 — mean μ = (1+3+5+7)/4 = {mu:.1f}")

# Step 2: variance（这里除以 N，不是 N-1）
var = torch.mean((x - mu) ** 2)
print(f"Step 2 — variance σ² = ((1-4)²+(3-4)²+(5-4)²+(7-4)²)/4")
print(f"        = (9 + 1 + 1 + 9)/4 = {var:.1f}")

# Step 3: std
sigma = torch.sqrt(var)
print(f"Step 3 — std σ = √{var:.1f} = {sigma:.4f}")

# Step 4: normalize
x_norm = (x - mu) / sigma
print(f"Step 4 — normalize: (x - 4)/{sigma:.4f}")
for i, (xi, xni) in enumerate(zip(x.tolist(), x_norm.tolist())):
    print(f"         x[{i}] = ({xi:.1f} - 4) / {sigma:.4f} = {xni: .4f}")

print(f"         normalize后: {[f'{v:.4f}' for v in x_norm.tolist()]}")
print(f"         mean: {x_norm.mean():.4f} (=0), std: {x_norm.std(unbiased=False):.4f} (=1)")

# Step 5: scale（假设 γ=[1,1,1,1], β=[0,0,0,0]）
# 初始时 γ 全 1，β 全 0，sooutput就是 x_norm
print(f"Step 5 — scale: γ=1, β=0 时output = normalize结果")
print(f"         γ 和 β 是随着training学出来的，让模型自己决定怎么调")

#### 1.2 The problem with LayerNorm: it computes something you may not need

LayerNorm computes two statistics:
- **mu (mean)**: shifts the center to 0
- **sigma (std)**: scales the spread

People tested: if we remove the mean-centering and only keep scaling, does it get worse?

Surprisingly: **not much**. Later linear layers (like W_Q, W_K, ...) can learn to handle non-zero-mean activations.

So why pay the cost of computing the mean every time? Removing it saves compute.

#### 1.3 RMSNorm: scaling only

Core formula:

```
RMS(x) = sqrt((x1^2 + x2^2 + x3^2 + x4^2) / 4)

RMSNorm(x) = x / RMS(x) * gamma
```

**Compare**:

```
LayerNorm: (x - mu) / sigma * gamma + beta   (two statistics: mu and sigma)
RMSNorm:    x / RMS(x) * gamma                (one statistic: RMS)
```

What you save:
- no mean
- no (x - mu)^2 term (you use x^2 directly)

Now compute RMSNorm on `[1, 3, 5, 7]` by hand.


In [ ]:
# === RMSNorm 手工compute ===
print("=== RMSNorm Compute by hand：input x = [1, 3, 5, 7] ===")
print()

x = torch.tensor([1.0, 3.0, 5.0, 7.0])

# 唯一的一步：算 RMS
mean_sq = torch.mean(x ** 2)
rms = torch.sqrt(mean_sq)

print(f"Step 1 — 平方mean = (1²+3²+5²+7²)/4")
print(f"          = (1+9+25+49)/4")
print(f"          = {84/4:.1f}")
print(f"    RMS = √{mean_sq:.1f} = {rms:.4f}")
print()

# RMSNorm output
x_rmsnorm = x / rms
print(f"Step 2 — RMSNorm = x / {rms:.4f}")
for i, (xi, xri) in enumerate(zip(x.tolist(), x_rmsnorm.tolist())):
    print(f"         x[{i}] = {xi:.1f} / {rms:.4f} = {xri:.4f}")

print(f"\nRMSNorm 结果: {[f'{v:.4f}' for v in x_rmsnorm.tolist()]}")

# 验证：RMSNorm 后的 RMS 应该是 1
rms_after = torch.sqrt(torch.mean(x_rmsnorm ** 2))
print(f"\n验证 — normalize后的 RMS = {rms_after:.4f} (应该 = 1) [ok]")
print(f"        Note：RMSNorm 后的mean ≠ 0（mean = {x_rmsnorm.mean():.4f}）")
print(f"        这就是和 LayerNorm 的区别——不去mean！")
print()

# 对比
ln = nn.LayerNorm(4, elementwise_affine=False)  # 关掉 γ,β 方便对比
x_ln = ln(x)
print(f"LayerNorm 结果: {[f'{v:.4f}' for v in x_ln.tolist()]}  mean={x_ln.mean():.4f}  std={x_ln.std(unbiased=False):.4f}")
print(f"RMSNorm  结果: {[f'{v:.4f}' for v in x_rmsnorm.tolist()]}  mean={x_rmsnorm.mean():.4f}  RMS={rms_after:.4f}")
print(f"\n-> 数值不同，但都起到了「标准化」的作用")
print(f"-> LayerNorm 强制mean=0，RMSNorm 不强制（但效果一样好）")

In [ ]:
# === RMSNorm 完整实现 ===

class RMSNorm(nn.Module):
    """
    RMSNorm: 只做scale，不做中心化
    
    公式: y = x / RMS(x) × γ
    RMS(x) = √(mean(x²) + ε)
    
    - 没有 β（bias），becausenot necessary补偿去mean造成的偏移
    - ε 是为了防止除以 0（比如input全 0）
    """
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))  # 只有 γ
    
    def forward(self, x):
        # x: [batch, seq_len, d_model]
        # 沿最后一维算 RMS
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.gamma


# 测试
d_model = 8
rmsn = RMSNorm(d_model)
ln = nn.LayerNorm(d_model)

batch = torch.randn(2, 4, d_model) * 3  # 模拟 2 个 batch, 4 个 token, 8 维
out_rms = rmsn(batch)
out_ln = ln(batch)

print("=== RMSNorm 实现验证 ===")
print(f"input形状: {batch.shape}")
print(f"RMSNorm output形状: {out_rms.shape}")
print(f"output RMS（应该 ≈ 1）: {torch.sqrt(torch.mean(out_rms**2, dim=-1))[0].tolist()}")
print()

# parameter(s)量对比
ln_params = sum(p.numel() for p in ln.parameters())
rmsn_params = sum(p.numel() for p in rmsn.parameters())
print(f"LayerNorm parameter(s): γ({d_model}) + β({d_model}) = {ln_params}")
print(f"RMSNorm  parameter(s): γ({d_model}) = {rmsn_params}")
print(f"-> parameter(s)减半，但这不是重点——重点是compute时少算了mean")

---

### 2. Upgrade 2: ReLU -> SwiGLU

#### 2.1 What does ReLU do in the original FFN?

Part-4 FFN:

```
FFN(x) = W2 * ReLU(W1 * x)

ReLU: negative -> 0, positive -> unchanged
```

Let's compute it for a concrete vector.


In [ ]:
# === ReLU Compute by hand ===
print("=== ReLU 对每个数字做了什么 ===")
print()

# 模拟 W₁·x 的output（扩展到 8 维）
hidden = torch.tensor([0.5, -2.0, 3.0, -0.1, 0.0, -5.0, 1.5, -0.01])

print(f"input (W₁·x 的结果): {[f'{v:.2f}' for v in hidden.tolist()]}")
print()

relu_out = F.relu(hidden)
print("ReLU 做了什么：")
for i, (h, r) in enumerate(zip(hidden.tolist(), relu_out.tolist())):
    action = "保留" if h >= 0 else "-> 0 (杀死)"
    print(f"  位置 {i}: {h:6.2f} -> {r:6.2f}  {action}")

print(f"\nIssue：8 个位置中有 {sum(1 for h in hidden if h < 0)} 个被直接杀死了")
print(f"      其中 -0.1 和 -0.01 只是「稍微负了一点」，也被杀了")
print(f"      -> ReLU 太粗暴：负的全部不要")

#### 2.2 Gating intuition: editor vs editor-in-chief

Imagine you write an article and submit it to an editor:

- **ReLU (one editor)**: if a paragraph is "negative" it gets deleted entirely. Even if it is 90% good and 10% flawed, it becomes 0. Wasteful.

- **Gating (two editors)**:
  - Editor A: understands content (information path)
  - Editor B: scores importance (gate path), output in [0, 1]
  - final output = A * B

So if something is "a bit wrong", B can give 0.3 and you keep part of it, instead of killing it completely.

Formula:

```
SwiGLU(x) = (W_up * x)  ⊙  Swish(W_gate * x)
              ^ info path     ^ gate path

Swish(a) = a * sigmoid(a) = a / (1 + e^{-a})
```

**What is Swish?** A smooth soft-switch, unlike ReLU's hard cutoff.


In [ ]:
# === ReLU vs Swish Compute by hand对比 ===
print("=== 激活函数对比：ReLU vs Swish ===")
print()

x_vals = [-3.0, -2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0]

print(f"{'x':>8s}  {'ReLU(x)':>10s}  {'Swish(x)':>10s}  {'说明'}")
print("-" * 52)

for x in x_vals:
    relu = max(0, x)
    swish = x / (1 + math.exp(-x))  # x * sigmoid(x)
    note = ""
    if x < 0:
        note = f"ReLU 杀死, Swish 保留 {swish:.4f}"
    elif x == 0:
        note = "ReLU 卡在 0, Swish 平滑"
    else:
        note = "两者都passes"
    print(f"{x:>8.1f}  {relu:>10.1f}  {swish:>10.4f}  {note}")

print()
print("Key observation：")
print("  1. 正数区: Swish ≈ ReLU（略小一点），行为相似")
print("  2. 负数区: ReLU = 0（彻底杀死），Swish 保留一个小负值")
print("  3. x=0 处: ReLU 不可导（数学上），Swish 光滑可导")
print()
print("Why「小负值」重要？")
print("  gradient = ∂loss/∂x，而 ∂Swish/∂x 在负数区不为 0")
print("  -> 即使这个神经元当前「不太激活」，gradient仍然可以流过")
print("  -> ReLU 的神经元可能「永久死亡」（一旦进入负数区，gradient=0，再也回不来）")

In [ ]:
# === 手动compute SwiGLU 的一步 ===
print("=== SwiGLU Compute by hand ===")
print()

# 用极小例子：d_model=4, 模拟一条信息
d_model = 4
x = torch.tensor([[1.0, -0.5, 2.0, 0.3]])  # 一个 token

# 模拟三个权重矩阵的简单版本（手工构造便于观察）
# 实际中这些是 nn.Linear 的权重
torch.manual_seed(1)
W_up = nn.Linear(d_model, d_model, bias=False)
W_gate = nn.Linear(d_model, d_model, bias=False)
W_down = nn.Linear(d_model, d_model, bias=False)

# 信息通道
up = W_up(x)
# 门控通道：先线性变换，再过 Swish
gate = F.silu(W_gate(x))  # silu = Swish
# 门控结果 = 信息 × 门
gated = up * gate
# output投影
out = W_down(gated)

print(f"input x: {x.tolist()}")
print()
print(f"信息通道 (W_up·x):     {[f'{v:.3f}' for v in up[0].tolist()]}")
print(f"门控通道 Swish(W_gate·x): {[f'{v:.3f}' for v in gate[0].tolist()]}")
print(f"门控后 (up × gate):     {[f'{v:.3f}' for v in gated[0].tolist()]}")
print(f"output (W_down·gated):    {[f'{v:.3f}' for v in out[0].tolist()]}")
print()
print("门控的效果: 如果 gate 某个位置 ≈ 0 -> 信息通道对应位置被「关门」-> 信息传不过去")
print("           如果 gate 某个位置 ≈ 1 -> 信息通道对应位置「开门」-> 信息完全passes")
print("           如果 gate 某个位置 = 0.3 -> 信息只passes 30%")

In [ ]:
# === 完整 SwiGLU FFN 实现 ===

class FeedForward_SwiGLU(nn.Module):
    """
    SwiGLU FFN（LLaMA / DeepSeek 同款）
    
    公式: FFN(x) = W_down · ( Swish(W_gate·x) ⊙ (W_up·x) )
    
    三个权重矩阵：
    - W_gate: 门控投影 (d_model -> d_ff)
    - W_up:   信息投影 (d_model -> d_ff)
    - W_down: output投影 (d_ff -> d_model)
    
    Note：原始 FFN 有 2 个矩阵，SwiGLU 有 3 个。为了保持parameter(s)量一致，
    d_ff 要调小：原始 d_ff=4d，SwiGLU d_ff ≈ 8/3 d
    """
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        if d_ff is None:
            # 3 * d_model * d_ff ≈ 2 * d_model * 4d_model
            # -> d_ff ≈ 8/3 * d_model
            d_ff = int(8 / 3 * d_model)
            d_ff = ((d_ff + 255) // 256) * 256  # 取整到 256 的倍数
            d_ff = max(d_ff, d_model)  # 至少和 d_model 一样大
        
        self.W_gate = nn.Linear(d_model, d_ff, bias=False)
        self.W_up = nn.Linear(d_model, d_ff, bias=False)
        self.W_down = nn.Linear(d_ff, d_model, bias=False)
    
    def forward(self, x):
        return self.W_down(F.silu(self.W_gate(x)) * self.W_up(x))


# parameter(s)量对比
d_model = 512
ffn_old = FeedForward_Old(d_model)
ffn_new = FeedForward_SwiGLU(d_model)

old_p = sum(p.numel() for p in ffn_old.parameters())
new_p = sum(p.numel() for p in ffn_new.parameters())

print("=== SwiGLU vs ReLU FFN parameter(s)量对比 ===")
print(f"d_model = {d_model}")
print()
print(f"ReLU FFN:  W1({d_model}×{4*d_model}) + W2({4*d_model}×{d_model})")
print(f"           = {d_model*4*d_model:,} + {4*d_model*d_model:,}")
print(f"           = {old_p:,} 个parameter(s)")
print()
d_ff_s = ((int(8/3*d_model) + 255) // 256) * 256
print(f"SwiGLU FFN: W_gate({d_model}×{d_ff_s}) + W_up({d_model}×{d_ff_s}) + W_down({d_ff_s}×{d_model})")
print(f"           = {d_model*d_ff_s:,} + {d_model*d_ff_s:,} + {d_ff_s*d_model:,}")
print(f"           = {new_p:,} 个parameter(s)")
print()
print(f"parameter(s)量比: {new_p/old_p:.2f}x (≈1 即可)")
print(f"-> parameter(s)差不多，但 SwiGLU 效果好很多")

---

### 3. Upgrade 3: Post-Norm -> Pre-Norm

#### 3.1 What does "Post" in Post-Norm mean?

In Part 4, the block is written like:

```python
x = x + attention(x)   # residual add
x = norm(x)            # LayerNorm
```

LayerNorm is applied **after** Attention, so it is called **Post-Norm**.

Computation graph (Post-Norm):

```
  x -> Attention -> + -> LayerNorm -> out
  |               ^
  +---------------+   (residual)
```

Notice: the residual path also goes through LayerNorm.

#### 3.2 Why is that a problem?

Residual connections are often described as "gradient highways": gradients can flow back without being weakened by Attention/FFN.

But in Post-Norm, the highway has a toll booth: **LayerNorm**.
LayerNorm rescales gradients.
In a 4-layer toy network it is fine, but in 40/80 layers the scaling can accumulate and cause exploding/vanishing gradients.

#### 3.3 Pre-Norm: move the toll booth off the highway

Pre-Norm moves normalization **before** Attention:

```python
x = x + attention(norm(x))
```

Graph (Pre-Norm):

```
  x -> LayerNorm -> Attention -> + -> out
  |                            ^
  +----------------------------+   (residual bypasses Norm)
```

Key change: the residual path bypasses normalization.
Now the gradient highway has no toll booth.

Next we compare two tiny networks.


In [ ]:
# === Post-Norm vs Pre-Norm：用Compute by hand看gradient流动 ===
print("=== Post-Norm vs Pre-Norm：gradient流动对比 ===")
print()

# 搭两个简化的深layer网络（用 FFN 代替 Attention，专注看 Norm 位置的影响）
d_model = 4
num_layers = 8  # 先用 8 layer看效果

class Deep_PostNorm(nn.Module):
    """Post-Norm: x = Norm(x + FFN(x))"""
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(d_model, d_model) for _ in range(num_layers)
        ])
        self.norms = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(num_layers)
        ])
    
    def forward(self, x):
        for layer, norm in zip(self.layers, self.norms):
            x = norm(x + F.relu(layer(x)))
        return x

class Deep_PreNorm(nn.Module):
    """Pre-Norm: x = x + FFN(Norm(x))"""
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(d_model, d_model) for _ in range(num_layers)
        ])
        self.norms = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(num_layers)
        ])
    
    def forward(self, x):
        for layer, norm in zip(self.layers, self.norms):
            x = x + F.relu(layer(norm(x)))
        return x

# 相同初始化
torch.manual_seed(42)
model_post = Deep_PostNorm(d_model, num_layers)
torch.manual_seed(42)
model_pre = Deep_PreNorm(d_model, num_layers)

# 随机input
x = torch.randn(2, d_model)
target = torch.randn(2, d_model)

# Forward + Backward
loss_post = F.mse_loss(model_post(x), target)
loss_post.backward()

loss_pre = F.mse_loss(model_pre(x), target)
loss_pre.backward()

# 看每一layer的gradient范数
print(f"网络深度: {num_layers} layer")
print()
print(f"{'layer':>6s}  {'Post-Norm gradient':>16s}  {'Pre-Norm gradient':>16s}")
print("-" * 42)

for i in range(num_layers):
    grad_post = model_post.layers[i].weight.grad.norm().item()
    grad_pre = model_pre.layers[i].weight.grad.norm().item()
    print(f"Layer {i+1}:  {grad_post:>16.6f}  {grad_pre:>16.6f}")

# 最重要：看第 1 layer的gradient（最底layer）
grad_post_first = model_post.layers[0].weight.grad.norm().item()
grad_pre_first = model_pre.layers[0].weight.grad.norm().item()

print()
print(f"最关键——最底layer（Layer 1）gradient对比:")
print(f"  Post-Norm Layer 1 gradient: {grad_post_first:.6f}")
print(f"  Pre-Norm  Layer 1 gradient: {grad_pre_first:.6f}")
print(f"  Pre-Norm / Post-Norm = {grad_pre_first/grad_post_first:.2f}x")
print()
print("-> Pre-Norm 底layergradient更大，becauseresidual路径绕过了所有 LayerNorm")
print("-> 在 80 layer网络里，这个差别会大到 Post-Norm 根本训不动")

#### 3.4 Cooking analogy summary

- **Post-Norm**: cook first, then taste. If you burn it, tasting later cannot fix it (the gradient got distorted).
- **Pre-Norm**: taste seasoning first, then cook. The signal stays stable throughout (residual path stays clean).

So for deep networks, Pre-Norm is much more stable. It tolerates larger learning rates and needs less warmup.


---

### 4. Put it together: a modern LLaMA-style block

With all three upgrades, the block becomes:

```
LLaMA Block = Pre-Norm (RMSNorm) + Attention + Pre-Norm (RMSNorm) + SwiGLU FFN

  x
  |
  +-> RMSNorm -> Attention -> +   <- residual (bypasses Norm)
  |                      |
  +----------------------+ 
  |
  +-> RMSNorm -> SwiGLU FFN -> + <- residual (bypasses Norm)
  |                         |
  +-------------------------+
  |
 out
```


In [ ]:
class LLaMABlock(nn.Module):
    """
    现代 LLM 的 Transformer Block
    
    三大升级 vs Part 4:
    1. LayerNorm -> RMSNorm
    2. Post-Norm -> Pre-Norm
    3. ReLU FFN -> SwiGLU FFN
    """
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        # Pre-Norm: RMSNorm 在 Attention 前面
        self.norm_attn = RMSNorm(d_model)
        # Multi-Head Self-Attention（和 Part 4 一样）
        self.attention = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        # Pre-Norm: RMSNorm 在 FFN 前面
        self.norm_ffn = RMSNorm(d_model)
        # SwiGLU FFN
        self.ffn = FeedForward_SwiGLU(d_model, d_ff)
    
    def forward(self, x, mask=None):
        # Note顺序：先 Norm，再子layer，再residual
        h = self.norm_attn(x)
        attn_out, _ = self.attention(h, h, h, attn_mask=mask, need_weights=False)
        x = x + attn_out  # residual：绕过 Norm！
        
        h = self.norm_ffn(x)
        x = x + self.ffn(h)  # residual：绕过 Norm！
        
        return x

# 测试
d_model, num_heads = 8, 2
block_new = LLaMABlock(d_model, num_heads)

test_x = torch.randn(1, 5, d_model)
causal_mask = torch.triu(torch.ones(5, 5) * float('-inf'), diagonal=1)

out_new = block_new(test_x, causal_mask)
print("=== LLaMA Block 测试 ===")
print(f"input形状: {test_x.shape}")
print(f"output形状: {out_new.shape}  ← 不变！")
print(f"但里面的组件全部升级了:")
print(f"  ✓ RMSNorm 替代 LayerNorm")
print(f"  ✓ Pre-Norm 替代 Post-Norm")
print(f"  ✓ SwiGLU 替代 ReLU FFN")

### 5. Evolution of Transformer blocks (three generations)

```
2017  Original Transformer (Vaswani et al.)
      Post-Norm + LayerNorm + ReLU FFN
      -> runs, but hard to train deep (the paper used 6 layers)

2019  GPT-2 / GPT-3
      Pre-Norm + LayerNorm + GELU FFN
      -> norm placement fixed, can stack ~96 layers
         GELU is better than ReLU, but not the end

2023  LLaMA 2/3, DeepSeek, Qwen, Mistral, ...
      Pre-Norm + RMSNorm + SwiGLU FFN
      -> all three upgrades, faster and more stable training
```

If we rank the importance:
1. **Pre-Norm** (most important: determines whether deep training is stable)
2. **SwiGLU** (large quality gains: gating makes FFN more expressive)
3. **RMSNorm** (nice-to-have: saves compute; often smaller impact)


---

## Architecture Refinements Summary

Check these in order:

1. [ok] LayerNorm = (x - mu) / sigma * gamma + beta (two statistics)
2. [ok] RMSNorm = x / RMS(x) * gamma (one statistic; no mean-centering)
3. [ok] RMSNorm saves compute: no mu, no (x - mu)^2 (use x^2)
4. [ok] ReLU issue: negative values are killed -> gradients can cut off (dead neurons)
5. [ok] SwiGLU = info path (W_up) * gate path (Swish(W_gate)) -> selective passing
6. [ok] Swish is smooth and has non-zero gradients for negative inputs
7. [ok] Post-Norm: sublayer + residual -> Norm (residual path passes through Norm)
8. [ok] Pre-Norm: Norm -> sublayer -> + residual (residual path bypasses Norm)
9. [ok] Pre-Norm stabilizes deep training (gradient highway has no toll booth)
10. [ok] LLaMA block = Pre-Norm + RMSNorm + SwiGLU (modern standard recipe)

**One-sentence summary**: Pre-Norm solves stability (can you train?), SwiGLU improves quality (how well can you train?), RMSNorm improves efficiency (how fast?).
